# V2D Track 2 on Colab: train + evaluate with the cluster's own scripts

Runtime: **A100 GPU + High-RAM** (Runtime -> Change runtime type). Needs `v2d_bundle_<prefix>.tar` in `MyDrive/v2d/` (built on the cluster by `colab/make_bundle.sh`).
Outputs and logs are written to `MyDrive/v2d/`, so a disconnect loses nothing already saved. Keep this tab open while runs train (no background execution on Colab Pro).
When finished: **Runtime -> Disconnect and delete runtime** (an idle GPU still burns units, ~8.9/h).

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Unpack the bundle and build flash_chord's exact environment (~5-10 min)
BUNDLE = '/content/drive/MyDrive/v2d/v2d_bundle_t2c.tar'
!test -f /content/v2d/BUNDLE_COMMIT || tar -xf {BUNDLE} -C /content
!BUNDLE={BUNDLE} DRIVE_DIR=/content/drive/MyDrive/v2d bash /content/v2d/colab/setup_colab.sh

In [ ]:
# 3. Smoke test: 2M steps + one evaluation (~5-8 min incl. kernel compile). Must end with an add_auc line.
!SMOKE=1 PREFIX=smoke MANIFEST=training_manifest_tier2c.json ARM=keypoints EP=4 bash /content/v2d/colab/run_colab.sh

In [ ]:
# 4. Start the real queue in the background: EP:ARM:SEED, LANES runs at a time on the one GPU
QUEUE = '4:keypoints:43 25:keypoints:43 28:keypoints:43 1:keypoints:43'
!PREFIX=t2c MANIFEST=training_manifest_tier2c.json LANES=2 QUEUE='{QUEUE}' nohup bash /content/v2d/colab/queue_colab.sh > /content/drive/MyDrive/v2d/logs/queue.out 2>&1 &
!sleep 5; cat /content/drive/MyDrive/v2d/logs/queue.out

In [ ]:
# 5. Watch progress every 10 min (keeps a cell busy while the queue runs; safe to stop and re-run)
import subprocess, time
STATUS = r'''date +%H:%M
nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv,noheader
for f in $(ls -t /content/v2d/logs/train_colab-*.out 2>/dev/null | head -4); do
  echo "== $(basename $f)"; tail -c 2000 "$f" | tr '\r' '\n' | grep -v '^\s*$' | tail -2
done
cat /content/drive/MyDrive/v2d/logs/queue.out'''
while subprocess.run(['pgrep', '-f', 'queue_colab.sh'], capture_output=True).returncode == 0:
    print(subprocess.run(STATUS, shell=True, capture_output=True, text=True).stdout, flush=True)
    time.sleep(600)
print(subprocess.run(STATUS, shell=True, capture_output=True, text=True).stdout)
print('queue finished')

In [ ]:
# 6. One tar of the results (policies + evaluations, no 152 MB training states) for the cluster
!cd /content/drive/MyDrive/v2d/outputs/cluster && tar --exclude='state_*' --exclude='replay_*' -cf /content/drive/MyDrive/v2d/results_$(date +%m%d_%H%M).tar t2c_* && ls -lh /content/drive/MyDrive/v2d/results_*.tar